To illustrate how to use stdio_client to list available tools

In [14]:
import sys
from mcp import StdioServerParameters

server_params = StdioServerParameters(
    command=sys.executable,
    args=[r"F:\projects\claude-client-sdk-example\src\mcpexample\weather_mcp_server\local_server.py"],
)

Define tool discovery using a session supplied by the caller. The final cell starts the server.

In [15]:
from mcp import ClientSession
from mcp.client.stdio import stdio_client
from pathlib import Path

project_root = Path(server_params.args[0]).resolve().parents[3]
log_dir = project_root / "logs"
log_dir.mkdir(exist_ok=True)

# Defining this function does not start a server.
# The caller supplies the session that it will use.
async def discover_tools(session):
    available_tools = await session.list_tools()
    print([tool.name for tool in available_tools.tools])
    return [
        {
            "name": tool.name,
            "description": tool.description or "",
            "input_schema": tool.input_schema,
        }
        for tool in available_tools.tools
    ]


Define the LLM request helper. The following helper executes its tool requests using the same MCP session.

In [16]:
import asyncio

# Make the project's util package available to this notebook.
src_dir = str(project_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from util.claude_client import get_claude_client_and_default_model

client, llm_model = get_claude_client_and_default_model(profile_name='claude')

async def ask_llm(messages, llm_tools, *, final_answer=False):
    options = {"tool_choice": {"type": "none"}} if final_answer else {}
    # Run the synchronous API client in a thread so MCP can keep processing I/O.
    response = await asyncio.to_thread(
        client.messages.create,
        model=llm_model,
        max_tokens=1000,
        messages=messages,
        tools=llm_tools,
        **options,
    )
    print(response)
    for block in response.content:
        if block.type == "tool_use":
            print("Requested tool:", block.name, "Arguments:", block.input)
        elif block.type == "text":
            print(block.text)
    return response


llm profile name : claude , model : claude-haiku-5-5


In [17]:
# Reuse the caller's session; this function never starts a server.
async def execute_tools(session, response):
    tool_results = []
    for call in response.content:
        if call.type != "tool_use":
            continue
        result = await session.call_tool(call.name, arguments=call.input)
        text = "\n".join(
            block.text for block in result.content if block.type == "text"
        )
        print(f"{call.name}:\n{text}\n")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": text,
            "is_error": result.is_error,
        })
    return tool_results


Run the cells above first.
This cell starts one server, shares one session across discovery and execution, and closes both when the block ends. Change the question below to try another request.
Not an elegant agent loop but shows how it works

In [18]:
messages = [{"role": "user", "content": "What are the current weather for NY?"}]

# Jupyter stderr needs a real file handle for the server's error output.
with (log_dir / "weather_mcp_local_stdio_server.log").open("a", encoding="utf-8") as server_log:
    # This is the only place in the notebook that starts an MCP server.
    async with stdio_client(server_params, errlog=server_log) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            llm_tools = await discover_tools(session)
            response = await ask_llm(messages, llm_tools)
            tool_results = await execute_tools(session, response)

            if tool_results:
                messages.extend([
                    {"role": "assistant", "content": [block.model_dump(exclude_none=True) for block in response.content]},
                    {"role": "user", "content": tool_results},
                ])
                final_response = await ask_llm(messages, llm_tools, final_answer=True)

# The server stops here, after all of the work above has finished.


['get_alerts', 'get_forecast']
Message(id='msg_011Cg85HLfU51akHwG9TvfnX', container=None, content=[ThinkingBlock(signature='CAQSgQcKEAgSGAI4AUIIdGhpbmtpbmcSDEWtP1TLkdqYfHbhKxoMMXB2AHOev3DOlsJ3IjAp0Omd1VmLdVTAlvLdi9ZO2lU1Ao3tKGh/MZWt0iDzOf4ckUy6HHPIU+oV9xLmrRoqngZBqYKoXryYOfxtdXXz66Ll8+cpH0LJKfkftVNdMARuI/uNuPOKEw2FeHTewwJz0EgcppxeL1Ab2r70bkQm1TrxcQKkPSnu1aA+jcSg2h/nJyfqr68kHhWFlxYeHuEja7whwnKXoZeu/9Yln6ZbasRXx1+TcoU2gu0oUtojrk3rHaS+8WB6q69pRvKdZCoV+CxPODk4vRJ4ciw05M6F2AGDHCaJOtoJZCmMD1eEl8zXRiafg0iFhxbnCF1EoyitlWYUnEl9++64zc0agurVyBfGE+Xdwk3Uebbie5QG33b+bgjeXTtJhGEFCCMq1p2EV5C3JXJmeGF9zdLfBGaEZhj7vwoDbf3VmPyXGmHI0er6QdR4zlaDPpLJ/0RLHIe00h03w8s9LMoJTRUgXa/nwR22VgZT+CdK7LkJ3B9A4+26SMx4K+RIavIgjZfy/xT1T+dwK+DHiB/XkRITaXmGXfLfclPSlALUmiBph3iPkiXNjqttIdRBSw/C+doVxKYDV5nJ2bhcCjPziFGADItT9c6YqatCZdomw0mBlneoRerF+vavQFmETx2Ib/6r2KR993NtGav+I3hJoWnK21EDw4RFRkrpXuu1GYcGitB/wmon+MH0Qfs7/ejLdOb5ICQlUOPGJYsODceGEyGNFw18I9cqs85IydqFDl7USPiRhbg8K8tvu0r1RVjD88tBF0CHMGq6E6OzDojPEQWOwzm24qw1WcLb5d4aadjZ